# What is Bayesian Optimization?

Bayesian Optimization is a smart, probabilistic method for finding the best hyperparameters for a model.

- It builds a probabilistic model of the objective function (e.g., accuracy).

- Uses previous results to decide which hyperparameter combinations to try next.

- Explores the search space efficiently, focusing on promising regions.

- Much faster than Grid Search for large or complex spaces.



## The Math Behind It

Suppose we have:

- Model $f(x;\theta)$ with parameters $\theta$.

- Hyperparameters $h = (h_1, h_2, ..., h_k)$.

- Objective function $CV(h)$ (e.g., cross-validation score).



Bayesian Optimization steps:

1. Build a surrogate model (e.g., Gaussian Process) to estimate $CV(h)$.

2. Use an acquisition function to select the next $h$ to evaluate (balance exploration and exploitation).

3. Evaluate $CV(h)$ for the selected $h$.

4. Update the surrogate model with the new result.

5. Repeat until stopping criteria are met.



Best hyperparameters:

  $$h^* = \arg\max_{h} CV(h)$$



## Example

Suppose we’re tuning an SVM:

- Hyperparameters:

  - Kernel = {linear, rbf}

  - C = [0.1, 10] (continuous range)

  - Gamma = [0.01, 0.1] (continuous range)



Bayesian Optimization will:

- Start with a few random settings.

- Build a model of how hyperparameters affect accuracy.

- Select new settings based on the model (not blindly).

- Use K-fold CV for evaluation.

- Pick best performing combination.







## Visual (Conceptual)

Imagine a landscape of scores for different hyperparameters.
- Bayesian Optimization builds a map of this landscape.
- It chooses where to “look” next based on where the best scores might be.



## Comparison with Grid Search CV

- **Grid Search CV:** Tries all combos blindly → exhaustive but slow.

- **Bayesian Optimization:** Explores smartly, learns from previous results, much faster for large spaces.


In [1]:
import pandas as pd

In [3]:
df=pd.read_csv("balanced_fraud_detection_data.csv")

In [5]:
from sklearn.model_selection import train_test_split

X = df.drop("is_fraud", axis=1)
y = df["is_fraud"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y)


In [ ]:
from sklearn.neighbors import KNeighborsClassifier
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier 
from sklearn.svm import SVC
from sklearn.linear_model import LogisticRegression

In [18]:
# Bayesian Optimization for KNN, Random Forest, and Logistic Regression
from skopt import BayesSearchCV
from skopt.space import Categorical

models = {
    'KNN': {
        'model': KNeighborsClassifier(),
        'params': {
            'n_neighbors': (3, 5, 7),# use (<value>) in bayes_search 
            'weights': ('uniform', 'distance'),
            'algorithm': ('auto', 'ball_tree'),
            'leaf_size': (30, 50)
        }
    },
    'RandomForest': {
        'model': RandomForestClassifier(random_state=42),
        'params': {
            'n_estimators': (100, 200, 300),
            'max_depth': (None, 10, 20),
            'min_samples_split': (2, 5),
            'criterion': ('gini', 'entropy')
        }
    },
    'LogisticRegression': {
        'model': LogisticRegression(solver='liblinear', random_state=42),
        'params': {
            'C': (0.1, 1, 10),
            'penalty': ('l1', 'l2'),
            'fit_intercept': (True, False),
            'max_iter': (100, 200)
        }
    }
}

bayes_results = {}
for name, mp in models.items():
    print(f"\nRunning Bayesian Optimization for {name}...")
    opt = BayesSearchCV(mp['model'], mp['params'], n_iter=32, cv=5, scoring='accuracy', n_jobs=-1, random_state=42)
    opt.fit(X_train, y_train)
    bayes_results[name] = {
        'best_score': opt.best_score_,
        'best_params': opt.best_params_,
        'test_score': opt.score(X_test, y_test)
    }
    print(f"Best CV Score: {opt.best_score_:.4f}")
    print(f"Best Params: {opt.best_params_}")
    print(f"Test Score: {opt.score(X_test, y_test):.4f}")

print("\nSummary of Bayesian Optimization Results:")
for name, res in bayes_results.items():
    print(f"{name}: Best CV Score={res['best_score']:.4f}, Test Score={res['test_score']:.4f}, Best Params={res['best_params']}")


Running Bayesian Optimization for KNN...
Best CV Score: 0.9280
Best Params: OrderedDict({'algorithm': 'auto', 'leaf_size': 30, 'n_neighbors': 7, 'weights': 'distance'})
Test Score: 0.9210

Running Bayesian Optimization for RandomForest...
Best CV Score: 0.9280
Best Params: OrderedDict({'algorithm': 'auto', 'leaf_size': 30, 'n_neighbors': 7, 'weights': 'distance'})
Test Score: 0.9210

Running Bayesian Optimization for RandomForest...


c:\Users\admin\anaconda3\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point ['entropy', None, 2, 100] before, using random point ['gini', None, 3, 300]
  warnings.warn(
c:\Users\admin\anaconda3\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point ['gini', 20, 4, 100] before, using random point ['entropy', 20, 2, 300]
  warnings.warn(
c:\Users\admin\anaconda3\Lib\site-packages\skopt\optimizer\optimizer.py:517: UserWarning: The objective has been evaluated at point ['gini', 20, 4, 100] before, using random point ['entropy', 20, 2, 300]
  warnings.warn(


Best CV Score: 0.9617
Best Params: OrderedDict({'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100})
Test Score: 0.9560

Running Bayesian Optimization for LogisticRegression...


c:\Users\admin\anaconda3\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension (True, False) was inferred to Categorical(categories=(True, False), prior=None). In upcoming versions of scikit-optimize, it will be inferred to <ValueError: the lower bound 1 has to be less than the upper bound 0>. See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\admin\anaconda3\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension (True, False) was inferred to Categorical(categories=(True, False), prior=None). In upcoming versions of scikit-optimize, it will be inferred to <ValueError: the lower bound 1 has to be less than the upper bound 0>. See the documentation of the check_dimension function for the upcoming API.
  warnings.warn(
c:\Users\admin\anaconda3\Lib\site-packages\skopt\space\space.py:116: UserWarning: Dimension (True, False) was inferred to Categorical(categories=(True, False), prior=None). In upcoming versions of s

Best CV Score: 0.8839
Best Params: OrderedDict({'C': 0.1, 'fit_intercept': True, 'max_iter': 200, 'penalty': 'l2'})
Test Score: 0.8740

Summary of Bayesian Optimization Results:
KNN: Best CV Score=0.9280, Test Score=0.9210, Best Params=OrderedDict({'algorithm': 'auto', 'leaf_size': 30, 'n_neighbors': 7, 'weights': 'distance'})
RandomForest: Best CV Score=0.9617, Test Score=0.9560, Best Params=OrderedDict({'criterion': 'entropy', 'max_depth': None, 'min_samples_split': 2, 'n_estimators': 100})
LogisticRegression: Best CV Score=0.8839, Test Score=0.8740, Best Params=OrderedDict({'C': 0.1, 'fit_intercept': True, 'max_iter': 200, 'penalty': 'l2'})
